# 15 · Ek CPU gecikme ölçümü ve teslim paketi

**Çalışma zamanı: CPU** (GPU gerekmez; Çalışma zamanı → Çalışma zamanı türünü değiştir → CPU). Süre yaklaşık 10-15 dk. Notebook 11-12 çalışmış olmalı (model ağırlıkları `outputs/runs/` içinde).

1. Ablasyon varyantları ve kontrol modeli için CPU gecikmesi (batch 1). Karşılaştırılabilirlik için tam model, ConvNeXt-Tiny ve DeiT-Small aynı oturumda yeniden ölçülür.
2. Dağıtım paketi: `deploy/` içinde float16 ağırlık kopyası (GitHub 100 MB sınırı için) ve float32 ile doğruluk kontrolü
3. `predict.py` denemesi (CPU)
4. Ortam kaydı (`pip freeze`, sürümler) ve çıktıların zip'lenmesi

In [ ]:
import os, sys, glob, shutil, zipfile
ON_KAGGLE = os.path.exists('/kaggle/working')

if ON_KAGGLE:
    # ---- Kaggle: kod ve önceki çıktılar /kaggle/input altındaki veri setlerinden gelir (salt okunur) ----
    hits = glob.glob('/kaggle/input/**/src/config.py', recursive=True)
    assert hits, 'Kod veri seti bulunamadı: sağ panelden Add Input ile yzproje-code veri setini ekleyin.'
    code_root = os.path.dirname(os.path.dirname(hits[0]))
    PROJECT = '/kaggle/working/YZProje'
    shutil.copytree(code_root, PROJECT, dirs_exist_ok=True)          # kod her oturumda tazelenir
    os.makedirs(PROJECT + '/outputs', exist_ok=True)
    out_hits = glob.glob('/kaggle/input/**/splits/all_files.csv', recursive=True)
    assert out_hits, 'Çıktı veri seti bulunamadı: Add Input ile yzproje-outputs veri setini ekleyin.'
    out_src = os.path.dirname(os.path.dirname(out_hits[0]))
    # yalnızca çalışma klasöründe OLMAYAN dosyaları kopyala (oturum içi yeni sonuçlar ezilmesin)
    shutil.copytree(out_src, PROJECT + '/outputs', dirs_exist_ok=True,
                    copy_function=lambda s, d: None if os.path.exists(d) else shutil.copy2(s, d))
    os.makedirs('/kaggle/temp/data', exist_ok=True)
    os.environ['KNEE_DATA_ROOT'] = '/kaggle/temp/data'                # ham veri: geçici disk (çıktıya sayılmaz)
else:
    # ---- Colab: Drive ----
    from google.colab import drive
    drive.mount('/content/drive')
    # Projeyi Drive'da bul (iç içe klasör olsa bile); zip açılmamışsa aç
    hits = glob.glob('/content/drive/MyDrive/**/src/config.py', recursive=True)
    if not hits:
        zips = glob.glob('/content/drive/MyDrive/**/YZProje*.zip', recursive=True)
        if zips:
            zipfile.ZipFile(zips[0]).extractall('/content/drive/MyDrive/')
            hits = glob.glob('/content/drive/MyDrive/**/src/config.py', recursive=True)
    if not hits:
        raise FileNotFoundError('Drive içinde src/config.py bulunamadı.')
    roots = [os.path.dirname(os.path.dirname(h)) for h in hits]
    # Birden fazla kopya varsa, veri listeleri (outputs/splits) olan klasör tercih edilir
    with_out = [r for r in roots if os.path.exists(f'{r}/outputs/splits/all_files.csv')]
    PROJECT = sorted(with_out or roots, key=len)[0]
    os.environ['KNEE_DATA_ROOT'] = '/content/data'

os.chdir(PROJECT); sys.path.insert(0, PROJECT)
os.environ['KNEE_OUTPUT_DIR'] = f'{PROJECT}/outputs'
print('Platform:', 'Kaggle' if ON_KAGGLE else 'Colab', '| Proje klasörü:', PROJECT)
print(sorted(os.listdir(PROJECT + '/outputs')) if os.path.exists(PROJECT + '/outputs') else 'outputs yok (ilk çalıştırma)')

In [ ]:
!pip install -q imagehash==4.3.2 tabulate==0.9.0 timm==1.0.30 scikit-image
import torch, platform
print(torch.__version__, platform.python_version(), 'GPU' if torch.cuda.is_available() else 'CPU')

In [ ]:
# Veri oturum diskinde yoksa yeniden indir (Colab oturumu kapanınca /content/data silinir)
DATA_URL = ''   # boş bırakabilirsiniz: 01 notebook'unda kullandığınız bağlantı Drive'daki kayıttan okunur
ZIP_PATH = ''
from pathlib import Path
from src import config as C, data_prep
if not (DATA_URL or ZIP_PATH):
    _info = Path(C.AUDIT_DIR) / 'download_info.json'
    if _info.exists():
        import json
        DATA_URL = json.load(open(_info)).get('download_url', '')
        if DATA_URL: print('Veri bağlantısı kayıttan okundu.')
if any(Path(C.EXTRACT_DIR).rglob('kneeKL224')) if Path(C.EXTRACT_DIR).exists() else False:
    print('Veri hazır:', data_prep.get_dataset_root())
else:
    if not (DATA_URL or ZIP_PATH):
        raise ValueError('Veri bu oturumda yok: DATA_URL veya ZIP_PATH doldurun.')
    # Sabit dosya listeleri Drive'da duruyor; yalnızca görüntüler yeniden açılır.
    data_prep.main(['--url', DATA_URL] if DATA_URL else ['--zip', ZIP_PATH])
    print('Veri hazır:', data_prep.get_dataset_root())

## 1 · CPU gecikme (ablasyon + kontrol)

In [ ]:
import os, platform, subprocess
print(platform.processor() or platform.machine(), '| CPU sayısı:', os.cpu_count())
print(subprocess.run(['bash','-c','lscpu | grep -E "Model name|Thread|Core"'], capture_output=True, text=True).stdout)
from src import final_eval as F
df = F.cpu_latency_extra()
df[['model','params','latency_ms_median','latency_ms_p95','round_medians']].round(1)

## 2 · Dağıtım paketi (float16 kopya + doğruluk kontrolü)

In [ ]:
from src import export
print(export.export())          # model.pt + meta.json (float32)
print(export.export_fp16())      # model_fp16.pt ve val üzerinde karşılaştırma

## 3 · predict.py denemesi

In [ ]:
from src.data import read_split
from src.data_prep import get_dataset_root
r = read_split('val').iloc[3]
!python predict.py "{get_dataset_root() / r.relpath}"
print('gerçek derece:', r.label)

## 4 · Ortam kaydı ve çıktıları kaydet

In [ ]:
import shutil, json, subprocess
from src import config as C
(C.OUTPUT_DIR / 'env').mkdir(parents=True, exist_ok=True)
(C.OUTPUT_DIR / 'env' / 'pip_freeze_final.txt').write_text(subprocess.run(['pip','freeze'], capture_output=True, text=True).stdout)
shutil.copytree(C.PROJECT_ROOT / 'deploy', '/content/deploy_out', dirs_exist_ok=True)
shutil.make_archive('/content/gecikme_ve_deploy', 'zip', '/content/deploy_out')
import pandas as pd
z = C.OUTPUT_DIR / 'final'
shutil.copy(z / 'cpu_latency_extra.csv', '/content/cpu_latency_extra.csv')
print('hazır: /content/gecikme_ve_deploy.zip ve /content/cpu_latency_extra.csv -> ikisini de indirip bana gönderin')
from google.colab import files
files.download('/content/cpu_latency_extra.csv'); files.download('/content/gecikme_ve_deploy.zip')

## 5 · Teslim klasörü (en son, rapor son hâlini aldıktan sonra)

Rapor dosyalarını (`YZProje_Rapor_*.docx/.pdf`) Drive'da `YZProje/report/` klasörüne koyun; sonra bu hücreyi çalıştırın. Çıktı: `22070090071_BahaTurk_YZProje.zip` (README'deki SHA256 otomatik doldurulur).

In [ ]:
!python tools/make_submission.py --project {C.PROJECT_ROOT} --out /content
from google.colab import files
files.download('/content/22070090071_BahaTurk_YZProje.zip')